In [104]:
import json
import os
from PIL import Image
import matplotlib.pyplot as plt
import requests
import re
import ast
import string
import textwrap
import base64
import random
import time
from openai import OpenAI
import io
import matplotlib.patches as mpatches

In [105]:
def generate_prompt(sample):
    question = sample["question_chart"]
    answer_type = sample["answer_type"]

    if answer_type == "numeric":
        format_rule = "a single number"
    elif answer_type == "numeric_list":
        format_rule = "[a, b, c]"
    elif answer_type == "choice":
        format_rule = "A"
    elif answer_type == "choice_list":
        format_rule = "[A, B, C]"
    elif answer_type == "table_list":
        format_rule = "[[...], [...], ...]"
    elif answer_type == "code":
        format_rule = "```python\n<code>\n```"
    else:
        format_rule = "short text"


    prompt = f"""
You are a data science assistant specialized in understanding charts and tables.

You are given a chart and a question.

Question:
{question}

Instructions:
1. Analyze the provided chart to answer the question.
2. Do NOT output reasoning, explanation, or intermediate steps.
3. Output ONLY the final answer.

Output format:
Answer: {format_rule}

Rules:
- Output exactly ONE line.
- The line must start with "Answer:".
- Do not output anything before or after the answer.
"""

    return prompt

In [106]:
def encode_image_to_base64(image_path, max_size=768, quality=70):
    img = Image.open(image_path)

    img.thumbnail((max_size, max_size))

    if img.mode != "RGB":
        img = img.convert("RGB")

    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=quality, optimize=True)

    return base64.b64encode(buffer.getvalue()).decode("utf-8")

In [ ]:
def call_llm(subject_path, sample):

    prompt = generate_prompt(sample)

    content_list = []

    chart_relative_path = sample["path"]["chart"].replace("\\", "/")
    chart_path = os.path.join(subject_path, chart_relative_path)

    if not os.path.exists(chart_path):
        raise FileNotFoundError(f"File does not exist: {chart_path}")

    image_base64 = encode_image_to_base64(chart_path)

    content_list.append({
        "type": "image_url",
        "image_url": {
            "url": f"data:image/jpeg;base64,{image_base64}"
        }
    })


    content_list.append({
        "type": "text",
        "text": prompt
    })

    client = OpenAI(
        base_url="",
        api_key="",
    )

    attempt = 0
    base_sleep=1.0
    max_sleep=60.0
    MAX_RETRY = 50

    while True:
        if attempt > MAX_RETRY:
            raise RuntimeError("LLM retry too many times")

        try:
            resp = client.chat.completions.create(
                model="",
                messages=[{"role": "user", "content": content_list}],
                temperature=0,
            )
            return resp.choices[0].message.content

        except Exception as e:
            print(f"error: {e}")
            msg = str(e).lower()

            if ("401" in msg) or ("403" in msg) or ("400" in msg) or ("invalid" in msg) or ("not found" in msg):
                raise

            sleep_s = min(max_sleep, base_sleep * (2 ** attempt))
            time.sleep(sleep_s)

            attempt += 1

In [ ]:
def exact_answer_llm(output, answer_type):

    if answer_type == "numeric":
        output_format = "answer: {number}. Return exactly one number after 'answer:'. Do not include units, symbols, commas, explanations, or extra text."

    elif answer_type == "numeric_list":
        output_format = "answer: [a, b, c]. Return a list of numbers inside square brackets. No text inside brackets. No explanations."

    elif answer_type == "choice":
        output_format = "answer: {A capital letter}. Return exactly one capital letter after 'answer:'. No punctuation. No brackets. No extra text."

    elif answer_type == "choice_list":
        output_format = "answer: [A, B, C]. Return a list of capital letters inside square brackets. No explanations."

    elif answer_type == "table_list":
        output_format = "answer: [[row1_col1, row1_col2, ...], [row2_col1, row2_col2, ...], ...]. Return a nested list representing the table. Only structured list format. "

    elif answer_type == "code":
        output_format = "answer: ```python {code block}```. Return only executable code inside the code block. No explanation outside the code block."

    else:
        output_format = "answer: {text}. Return only the final short text answer after 'answer:'. The answer MUST end with a period '.'. Do not include anything after the period. No explanations or additional commentary."


    prompt = f"""
You are an answer extraction assistant. Your task is to extract the final answer from a model’s response according to the specified answer type.
Answer type: {answer_type}
Model response:
{output}
You must extract only the final answer and strictly follow this output format:
{output_format}
Do not include explanations or any additional text.
Return exactly one answer in the required format.
    """

    client = OpenAI(
        base_url="",
        api_key="",
    )

    attempt = 0
    base_sleep=1.0
    max_sleep=60.0
    MAX_RETRY = 50

    while True:
        if attempt > MAX_RETRY:
            raise RuntimeError("LLM retry too many times")

        try:
            response = client.chat.completions.create(
                model="deepseek-chat",
                messages=[
                    {"role": "system", "content": prompt}
            ],
                max_tokens=1024,
                temperature=0,
                stream=False
            )

            output_llm = response.choices[0].message.content

            if not output_llm:
                raise ValueError("empty response")

            return output_llm

        except Exception as e:
            msg = str(e).lower()

            if ("401" in msg) or ("403" in msg) or ("400" in msg) or ("invalid" in msg) or ("not found" in msg):
                raise

            sleep_s = min(max_sleep, base_sleep * (2 ** attempt))
            time.sleep(sleep_s)

            attempt += 1

In [110]:

def evaluate_numeric(output, gt):
    eps_rel = 0.01
    eps_abs = 1e-6

    if output is None:
        return False

    output = output.strip()

    output_llm = exact_answer_llm(output, "numeric")

    number_pattern = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"

    match = re.search(rf"^answer:\s*({number_pattern})\s*$", output_llm.strip(), re.IGNORECASE)

    if match:
        pred = float(match.group(1))
    else:
        match = re.search(r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?", output_llm)

        if not match:
            return False

        pred = float(match.group())

    gt = float(gt)

    tolerance = max(eps_abs, eps_rel * abs(gt))

    return abs(pred - gt) <= tolerance

In [111]:

def evaluate_numeric_list(output, gt):
    eps_rel = 0.01
    eps_abs = 1e-6

    if output is None:
        return False

    s = output.strip()

    output_llm = exact_answer_llm(s, "numeric_list")

    number_pattern = r"[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?"
    match = re.search(rf"(?is)answer:\s*(\[\s*{number_pattern}(?:\s*,\s*{number_pattern})*\s*\])", output_llm)

    if match:
        list_str = match.group(1)
    else:
        match = re.search(rf"(\[\s*{number_pattern}(?:\s*,\s*{number_pattern})*\s*\])", output_llm)
        if match:
            list_str = match.group(1)
        else:
            return False

    try:
        pred = ast.literal_eval(list_str)
    except Exception:
        return False

    if not isinstance(pred, list):
        return False

    if len(pred) != len(gt):
        return False

    for pi, gi in zip(pred, gt):
        try:
            pi = float(pi)
            gi = float(gi)
        except Exception:
            return False

        toli = max(eps_abs, eps_rel * abs(gi))
        if abs(pi - gi) > toli:
            return False

    return True


In [112]:

def normalize_text(s):
    if s is None:
        return ""

    s = str(s).lower().strip()

    punct = string.punctuation + "，。！？；：“”‘’（）【】《》、…—·"
    s = s.translate(str.maketrans("", "", punct))

    s = re.sub(r"\s+", " ", s).strip()
    return s

def edit_distance(a, b):
    n, m = len(a), len(b)
    if n == 0:
        return m
    if m == 0:
        return n

    prev = list(range(m + 1))
    for i in range(1, n + 1):
        cur = [i] + [0] * m
        ai = a[i - 1]
        for j in range(1, m + 1):
            cost = 0 if ai == b[j - 1] else 1
            cur[j] = min(
                prev[j] + 1,
                cur[j - 1] + 1,
                prev[j - 1] + cost
            )
        prev = cur
    return prev[m]

def evaluate_text(output, gt):
    threshold = 0.9

    if output is None:
        return False

    output_llm = exact_answer_llm(output, "text")

    m = re.search(r"(?is)^\s*answer:\s*(.+?\.)", output_llm.strip())
    if not m:
        return False

    pred_text = m.group(1).strip()

    p = normalize_text(pred_text)
    g = normalize_text(gt)

    if len(p) == 0 and len(g) == 0:
        return True
    if len(p) == 0 or len(g) == 0:
        return False

    dist = edit_distance(p, g)
    sim = 1.0 - dist / max(len(p), len(g))

    return sim >= threshold



In [113]:

def evaluate_choice(output, gt):
    s = str(output).strip()

    output_llm = exact_answer_llm(s, "choice")

    match = re.search(r"(?is)^\s*answer:\s*([A-Z])\s*(?:\r?\n|$)", output_llm.strip())
    if not match:
        return False

    pred = match.group(1)

    return pred == str(gt).strip()


In [ ]:
ROOT_PATH = "table_and_chart"
JSON_NAME = "question.json"

full_json_path = os.path.join(ROOT_PATH, JSON_NAME)


def evaluate_dataset(subject_path, full_json_path):
    with open(full_json_path, "r", encoding="utf-8") as f:
        samples_data = json.load(f)

    stats = {
        "perceptual": {"score": 0, "count": 0},
        "reasoning": {"score": 0, "count": 0},
        "table to chart": {"score": 0, "count": 0},
        "chart to table": {"score": 0, "count": 0},
    }

    for sample in samples_data:
        if sample["answer_type"] in ["code", "table_list"]:
            continue

        score = 0

        print("-----sample-----")
        print(sample)

        output = call_llm(subject_path, sample)

        answer_type = sample["answer_type"]
        gt = sample["answer"]

        if answer_type == "numeric":
            score = evaluate_numeric(output, gt)
        elif answer_type == "numeric_list":
            score = evaluate_numeric_list(output, gt)
        elif answer_type == "text":
            score = evaluate_text(output, gt)

        question_type = sample["question_type"]

        if question_type in stats:
            stats[question_type]["count"] += 1
            stats[question_type]["score"] += score

    return stats


if not os.path.exists(full_json_path):
    raise FileNotFoundError(f"Does not exist: {full_json_path}")

print(f"\n====== Running {full_json_path} ======")

global_stats = evaluate_dataset(ROOT_PATH, full_json_path)


print("\n====== Result ======")

for k in global_stats:
    total_score = global_stats[k]["score"]
    total_count = global_stats[k]["count"]
    avg = total_score / total_count if total_count else 0

    print(f"{k}: Total Count={total_count}, Avg={avg:.4f}")